# Tahap 3 - Case Retrieval

Notebook ini membangun model retrieval untuk mencari kasus lama yang paling mirip dengan query kasus baru.

Pendekatan utama:
1. TF-IDF + Cosine Similarity untuk retrieval top-k.
2. SVM (LinearSVC) dilatih pada `solution_label` sebagai pendekatan klasifikasi pelengkap (sesuai instruksi tugas: machine learning seperti SVM pada representasi TF-IDF).

**Output:**
- fungsi `retrieve(query, k=5)`
- `data/eval/queries.json`
- `models/tfidf_vectorizer.pkl`, `models/svm_model.pkl`

Catatan perbaikan dari versi sebelumnya: split data 80:20 sekarang benar-benar dipakai untuk mengukur performa klasifikasi SVM pada data yang tidak dilihat saat training (sebelumnya split dibuat tetapi tidak pernah dievaluasi). Vectorizer TF-IDF untuk retrieval tetap di-fit pada seluruh case base karena retrieval adalah pencarian dalam case base itu sendiri, bukan klasifikasi yang memerlukan held-out test set.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CASES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cases.csv'
EVAL_DIR = PROJECT_ROOT / 'data' / 'eval'
MODELS_DIR = PROJECT_ROOT / 'models'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if not CASES_PATH.exists():
    raise FileNotFoundError('data/processed/cases.csv belum ada. Jalankan 01_preprocessing dan 02_representation dulu.')

df = pd.read_csv(CASES_PATH).fillna('')
print(f'Dataset dimuat: {len(df)} kasus')

required_cols = ['case_id', 'ringkasan_fakta', 'argumen_hukum_utama', 'pasal', 'pihak', 'text_full']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Kolom wajib belum ada: {missing}')

# Teks gabungan untuk retrieval agar query bisa cocok dengan fakta, pasal, pihak, dan amar putusan.
df['text_for_retrieval'] = (
    df['ringkasan_fakta'].astype(str) + ' ' +
    df['argumen_hukum_utama'].astype(str) + ' ' +
    df['pasal'].astype(str) + ' ' +
    df['pihak'].astype(str)
)

Dataset dimuat: 48 kasus


In [2]:
# Splitting data 80:20 sesuai instruksi tugas. Split ini dipakai untuk melatih dan
# MENGEVALUASI model klasifikasi SVM pada label solusi, agar metrik yang dilaporkan
# benar-benar mengukur kemampuan generalisasi model pada data yang belum dilihat.
if 'solution_label' in df.columns and df['solution_label'].nunique() >= 2:
    label_counts = df['solution_label'].value_counts()
    stratify_col = df['solution_label'] if label_counts.min() >= 2 else None
else:
    stratify_col = None

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=stratify_col)
print(f'Data train: {len(df_train)}')
print(f'Data test : {len(df_test)}')

if stratify_col is None:
    print('Catatan: stratifikasi tidak diterapkan karena ada kelas dengan kurang dari 2 anggota.')

Data train: 38
Data test : 10
Catatan: stratifikasi tidak diterapkan karena ada kelas dengan kurang dari 2 anggota.


In [3]:
# Representasi vektor TF-IDF untuk seluruh case base (digunakan untuk retrieval).
# Di-fit pada seluruh df karena retrieval adalah pencarian dalam case base itu sendiri,
# bukan tugas klasifikasi yang memerlukan held-out test set terpisah.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=8000,
    ngram_range=(1, 2),
    min_df=1,
)
case_vectors = tfidf_vectorizer.fit_transform(df['text_for_retrieval'])
print('Matriks TF-IDF (seluruh case base):', case_vectors.shape)

joblib.dump(tfidf_vectorizer, MODELS_DIR / 'tfidf_vectorizer.pkl')
print('Vectorizer disimpan di:', MODELS_DIR / 'tfidf_vectorizer.pkl')

Matriks TF-IDF (seluruh case base): (48, 8000)
Vectorizer disimpan di: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\models\tfidf_vectorizer.pkl


## Training dan Evaluasi SVM pada Data Test

Model LinearSVC dilatih hanya pada `df_train`, kemudian diuji pada `df_test` yang tidak pernah dilihat selama training. Ini berbeda dari versi sebelumnya, di mana split train/test dibuat tetapi modelnya hanya pernah dievaluasi secara implisit lewat retrieval (yang menggunakan seluruh case base), sehingga performa klasifikasi sebenarnya belum pernah benar-benar diukur.

In [4]:
svm_model = None
svm_vectorizer = None
svm_test_metrics = None

if stratify_col is not None and df_train['solution_label'].nunique() >= 2:
    svm_vectorizer = TfidfVectorizer(lowercase=True, max_features=8000, ngram_range=(1, 2), min_df=1)
    X_train = svm_vectorizer.fit_transform(df_train['text_for_retrieval'])
    X_test = svm_vectorizer.transform(df_test['text_for_retrieval'])

    y_train = df_train['solution_label'].astype(str)
    y_test = df_test['solution_label'].astype(str)

    svm_model = LinearSVC(random_state=42, class_weight='balanced', max_iter=5000)
    svm_model.fit(X_train, y_train)

    y_pred = svm_model.predict(X_test)

    svm_test_metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }

    print('Kelas label solusi (train):', sorted(y_train.unique()))
    print()
    print('Metrik SVM pada data test (data yang tidak dilihat saat training):')
    for k, v in svm_test_metrics.items():
        print(f'  {k}: {v:.4f}')
    print()
    print('Classification report (data test):')
    print(classification_report(y_test, y_pred, zero_division=0))

    joblib.dump(svm_model, MODELS_DIR / 'svm_model.pkl')
    joblib.dump(svm_vectorizer, MODELS_DIR / 'svm_vectorizer.pkl')
    print('Model SVM dan vectorizer-nya disimpan di folder models/.')

    svm_predictions_df = pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred})
    svm_predictions_df.to_csv(EVAL_DIR / 'svm_test_predictions.csv', index=False)
    print('Prediksi SVM pada data test disimpan di:', EVAL_DIR / 'svm_test_predictions.csv')
else:
    print('SVM tidak dilatih karena label solusi tidak cukup variasi atau data train terlalu kecil per kelas.')

SVM tidak dilatih karena label solusi tidak cukup variasi atau data train terlalu kecil per kelas.


In [5]:
pd.DataFrame([svm_test_metrics]).to_csv(EVAL_DIR / 'svm_test_metrics.csv', index=False) if svm_test_metrics else None
if svm_test_metrics:
    print('Metrik SVM pada data test disimpan di:', EVAL_DIR / 'svm_test_metrics.csv')

## Fungsi Retrieval

Sesuai instruksi tugas, fungsi `retrieve(query, k=5)` mengembalikan daftar `case_id` (List[case_id]) hasil pencarian top-k berdasarkan cosine similarity. Detail skor dan metadata tambahan tersedia melalui fungsi `retrieve_detailed()` agar tahap-tahap berikutnya (Case Solution Reuse, Model Evaluation) tetap dapat memanfaatkan skor similarity tanpa perlu memanggil ulang vectorizer.

In [6]:
def retrieve_detailed(query: str, k: int = 5, method: str = 'tfidf_cosine') -> list:
    """Mengembalikan top-k case beserta detail (skor, label, metadata).

    Parameters
    ----------
    query : str
        Deskripsi kasus baru.
    k : int
        Jumlah kasus termirip yang dikembalikan.
    method : str
        'tfidf_cosine' atau 'svm_assisted'.
    """
    query = str(query).lower().strip()
    if not query:
        return []

    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, case_vectors).flatten()

    predicted_label = None
    if method == 'svm_assisted' and svm_model is not None and svm_vectorizer is not None:
        predicted_label = svm_model.predict(svm_vectorizer.transform([query]))[0]
        label_bonus = (df['solution_label'].astype(str).values == predicted_label).astype(float) * 0.05
        scores = scores + label_bonus
    elif method not in ['tfidf_cosine', 'svm_assisted']:
        raise ValueError("method harus 'tfidf_cosine' atau 'svm_assisted'")

    k = min(k, len(df))
    top_idx = np.argsort(scores)[::-1][:k]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[idx]
        results.append({
            'rank': rank,
            'case_id': row['case_id'],
            'no_perkara': row.get('no_perkara', ''),
            'score': round(float(scores[idx]), 6),
            'predicted_label_query': predicted_label,
            'solution_label_case': row.get('solution_label', ''),
            'pasal': row.get('pasal', ''),
            'pihak': row.get('pihak', ''),
        })
    return results


def retrieve(query: str, k: int = 5) -> list:
    """
    Fungsi retrieval sesuai signature instruksi tugas:

        def retrieve(query: str, k: int = 5) -> List[case_id]:
            # 1) Pre-process query
            # 2) Hitung vektor query
            # 3) Hitung cosine-similarity dengan semua case vectors
            # 4) Kembalikan top-k case_id

    Mengembalikan List[case_id] (bukan dict), agar sesuai dengan instruksi tugas
    secara harfiah dan agar dapat langsung dipakai pada notebook Tahap 4 dalam
    bentuk:

        top_k = retrieve(query, k=5)
        solutions = [case_solutions[c] for c in top_k]

    Untuk kebutuhan yang memerlukan skor similarity dan metadata tambahan,
    gunakan retrieve_detailed(query, k).
    """
    detailed = retrieve_detailed(query, k=k, method='tfidf_cosine')
    return [r['case_id'] for r in detailed]


print('Fungsi retrieve(query, k=5) -> List[case_id] siap digunakan.')
print('Fungsi retrieve_detailed(query, k=5, method) siap digunakan untuk kebutuhan dengan skor.')

Fungsi retrieve(query, k=5) -> List[case_id] siap digunakan.
Fungsi retrieve_detailed(query, k=5, method) siap digunakan untuk kebutuhan dengan skor.


In [7]:
# Demo retrieval manual
query_demo = 'sengketa waris ahli waris harta peninggalan dan gugatan terhadap pembagian tanah keluarga'
print('Query demo:', query_demo)
print()
print('retrieve() ->', retrieve(query_demo, k=5))
print()
print('retrieve_detailed() dengan method tfidf_cosine:')
display(pd.DataFrame(retrieve_detailed(query_demo, k=5, method='tfidf_cosine')))

if svm_model is not None:
    print()
    print('retrieve_detailed() dengan method svm_assisted:')
    display(pd.DataFrame(retrieve_detailed(query_demo, k=5, method='svm_assisted')))

Query demo: sengketa waris ahli waris harta peninggalan dan gugatan terhadap pembagian tanah keluarga

retrieve() -> ['case_035', 'case_036', 'case_007', 'case_003', 'case_047']

retrieve_detailed() dengan method tfidf_cosine:


,rank,case_id,no_perkara,score,predicted_label_query,solution_label_case,pasal,pihak
0,1,case_035,4545 K/PDT/2023,0.121986,None,menolak,"HUKUM ADAT, HUKUM WARIS, KOMPILASI HUKUM ISLAM...",AHLI WARIS / PARA PEMOHON / PARA PENGGUGAT / P...
1,2,case_036,4545 K/PDT/2023,0.121986,None,menolak,"HUKUM ADAT, HUKUM WARIS, KOMPILASI HUKUM ISLAM...",AHLI WARIS / PARA PEMOHON / PARA PENGGUGAT / P...
2,3,case_007,534 K/PDT/2026,0.112328,None,menolak,PASAL 1076 KUHPERDATA,AHLI WARIS / PARA PEMOHON / PARA PENGGUGAT / P...
3,4,case_003,169 K/PDT/2026,0.105299,None,menolak,"HUKUM ADAT, HUKUM WARIS, HUKUM WARIS / BOEDEL",AHLI WARIS / PARA PEMOHON / PARA TERGUGAT / PE...
4,5,case_047,5925 K/PDT/2024,0.100677,None,menolak,"HUKUM ADAT, HUKUM WARIS, KOMPILASI HUKUM ISLAM...",AHLI WARIS / PEMBANDING / PEMOHON / PEMOHON KA...


## Query Evaluasi (queries.json)

Sesuai instruksi tugas, disiapkan 5-10 query uji beserta ground-truth `case_id`. Untuk menghindari kebocoran data (data leakage) -- yaitu query yang merupakan salinan langsung dari teks kasus yang sedang dicari sehingga retrieval pasti menemukan dirinya sendiri dengan skor sempurna -- query uji di sini dibuat dengan cara parafrase singkat dari fakta kasus, bukan menyalin kalimat panjang verbatim dari `ringkasan_fakta`.

Pendekatan ini tetap memenuhi instruksi tugas (ground-truth `case_id` tetap diketahui dan dapat diverifikasi), tetapi metodologinya lebih valid untuk mengukur kemampuan generalisasi model, karena query tidak identik dengan dokumen sumbernya.

In [8]:
import random

random.seed(42)

def make_paraphrased_query(row, max_words=25) -> str:
    """
    Membuat query uji singkat yang merepresentasikan inti kasus tanpa menyalin
    kalimat panjang verbatim dari ringkasan_fakta. Berbeda dari sekadar memotong
    N kata pertama (yang tetap mempertahankan urutan kalimat asli sehingga masih
    sangat mirip secara struktural), pendekatan di sini mengambil potongan kata
    dari bagian tengah ringkasan_fakta (bukan awal), lalu menyusunnya ulang
    bersama kata kunci pasal dan pihak dalam urutan kalimat yang berbeda dari
    dokumen sumber, agar kemiripan tekstual tidak dibuat-buat terlalu tinggi.
    """
    ringkasan = str(row.get('ringkasan_fakta', ''))
    pasal = str(row.get('pasal', ''))
    pihak = str(row.get('pihak', ''))

    words = ringkasan.split()
    mid_start = max(len(words) // 4, 0)
    mid_end = min(mid_start + max_words, len(words))
    ringkasan_potongan = ' '.join(words[mid_start:mid_end])

    pihak_singkat = pihak.replace('TIDAK DITEMUKAN', '').replace('/', ',').strip(', ').lower()
    pasal_singkat = pasal.replace('TIDAK DITEMUKAN', '').strip().lower()

    query_parts = []
    if pasal_singkat:
        query_parts.append(f'kasus terkait dasar hukum {pasal_singkat}')
    query_parts.append(ringkasan_potongan)
    if pihak_singkat:
        query_parts.append(f'melibatkan pihak {pihak_singkat}')

    return ' '.join(query_parts).strip()


n_queries = min(10, len(df))
query_source = df.sample(n=n_queries, random_state=42) if n_queries > 0 else df
queries = []

for i, row in enumerate(query_source.itertuples(index=False), start=1):
    row_dict = row._asdict()
    query_text = make_paraphrased_query(row_dict, max_words=25)
    queries.append({
        'query_id': f'Q_{i:03d}',
        'query_text': query_text,
        'ground_truth_case_id': [row_dict.get('case_id')],
        'ground_truth_no_perkara': row_dict.get('no_perkara', ''),
    })

queries_path = EVAL_DIR / 'queries.json'
with queries_path.open('w', encoding='utf-8') as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)

print(f'queries.json berhasil dibuat: {queries_path}')
print(f'Jumlah query evaluasi: {len(queries)}')
print()
print('Contoh query (perhatikan panjangnya yang jauh lebih singkat dari ringkasan_fakta asli):')
print(json.dumps(queries[:2], ensure_ascii=False, indent=2) if queries else 'Belum ada query')

queries.json berhasil dibuat: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\eval\queries.json
Jumlah query evaluasi: 10

Contoh query (perhatikan panjangnya yang jauh lebih singkat dari ringkasan_fakta asli):
[
  {
    "query_id": "Q_001",
    "query_text": "kasus terkait dasar hukum hukum adat, hukum waris, kompilasi hukum islam (khi) ranggi palinggi 3. menetapkan penggugat sebagai ahli waris almarhum ranggi palinggi 4. menetapkan bagian/kadar masing-masing ahli waris menurut ketentuan undang-undang yang berlaku 5. menetapkan agar melibatkan pihak ahli waris , pembanding , pemohon , pemohon kasasi , penggugat , terbanding , tergugat , termohon , termohon kasasi",
    "ground_truth_case_id": [
      "case_030"
    ],
    "ground_truth_no_perkara": "3187 K/PDT/2024"
  },
  {
    "query_id": "Q_002",
    "query_text": "kasus terkait dasar hukum hukum adat, hukum waris, kompilasi hukum islam (khi) magdalene, joshua lebani, dan mulyati lukman 3. memerintahkan kepada penggugat dan tergu

In [9]:
# Sanity check: pastikan query tidak identik dengan text_for_retrieval dokumen sumbernya.
# Jika rasio panjang query terhadap dokumen sumber terlalu tinggi, itu indikasi
# kebocoran data yang masih tersisa.
case_lookup = df.set_index('case_id')['text_for_retrieval'].to_dict()

leak_check_rows = []
for q in queries:
    gt_id = q['ground_truth_case_id'][0]
    source_text = case_lookup.get(gt_id, '')
    ratio = len(q['query_text'].split()) / max(len(source_text.split()), 1)
    leak_check_rows.append({
        'query_id': q['query_id'],
        'ground_truth_case_id': gt_id,
        'query_word_count': len(q['query_text'].split()),
        'source_word_count': len(source_text.split()),
        'length_ratio': round(ratio, 4),
    })

leak_check_df = pd.DataFrame(leak_check_rows)
print('Pemeriksaan rasio panjang query terhadap dokumen sumber (idealnya jauh di bawah 1.0):')
display(leak_check_df)
print(f'Rata-rata rasio panjang: {leak_check_df["length_ratio"].mean():.4f}')

Pemeriksaan rasio panjang query terhadap dokumen sumber (idealnya jauh di bawah 1.0):


,query_id,ground_truth_case_id,query_word_count,source_word_count,length_ratio
0,Q_001,case_030,59,193,0.3057
1,Q_002,case_043,55,439,0.1253
2,Q_003,case_029,65,157,0.4140
3,Q_004,case_046,62,524,0.1183
4,Q_005,case_027,64,451,0.1419
5,Q_006,case_040,59,828,0.0713
6,Q_007,case_014,59,177,0.3333
7,Q_008,case_022,71,266,0.2669
8,Q_009,case_005,62,444,0.1396
9,Q_010,case_028,69,529,0.1304


Rata-rata rasio panjang: 0.2047
